# Datenvalidierung mit Voluptuous (Schemadefinitionen)

In diesem Notebook verwenden wir [Voluptuous](https://github.com/alecthomas/voluptuous), um Schemata für unsere Daten zu definieren. Wir können dann die Schemaprüfung an verschiedenen Stellen unserer Bereinigung verwenden, um sicherzustellen, dass wir die Kriterien erfüllen. Schließllich können wir Ausnahmen für die Schemaüberprüfung verwenden, um unreine oder ungültige Daten zu markieren, beiseite zu legen oder zu entfernen.

<div class="alert alert-block alert-info">

**Siehe auch**

* [Validr](https://github.com/guyskk/validr)
* [marshmallow](https://marshmallow.readthedocs.io/en/latest/)
</div>

## 1. Importe

In [1]:
import datetime
import logging

import pandas as pd

from voluptuous import ALLOW_EXTRA, All, Range, Required, Schema
from voluptuous.error import Invalid, MultipleInvalid

* `Required` markiert den Knoten eines Schemas als erforderlich und gibt optional einen Standardwert an, siehe auch [voluptuous.schema_builder.Required](http://alecthomas.github.io/voluptuous/docs/_build/html/voluptuous.html?highlight=required#voluptuous.schema_builder.Required).
* `Range` begrenzt den Wert auf einen Bereich, wobei entweder `min` oder `max` weggelassen werden kann; siehe auch [voluptuous.validators.Range](http://alecthomas.github.io/voluptuous/docs/_build/html/voluptuous.html?highlight=required#voluptuous.validators.Range).
* `ALL` wird für feldübergreifende Validierungen verwendet: prüft die Grundstruktur der Daten in einem ersten Durchgang und erst im zweiten Durchgang wird die feldübergreifende Validierung angewendet; siehe auch [voluptuous.validators.All](http://alecthomas.github.io/voluptuous/docs/_build/html/voluptuous.html?highlight=required#voluptuous.validators.All).
* `ALLOW_EXTRA` erlaubt zusätzliche Wörterbuchschlüssel
* `MultipleInvalid` basiert auf `Invalid`, siehe auch [voluptuous.error.MultipleInvalid](http://alecthomas.github.io/voluptuous/docs/_build/html/voluptuous.html?highlight=required#voluptuous.error.MultipleInvalid).
* `Invalid` kennzeichnet Daten als ungültig, siehe auch [voluptuous.error.Invalid](http://alecthomas.github.io/voluptuous/docs/_build/html/voluptuous.html?highlight=required#voluptuous.error.Invalid).

## 2. Logger

In [2]:
logger = logging.getLogger(__name__)

## 3. Beispieldaten lesen

In [3]:
sales = pd.read_csv(
    "https://raw.githubusercontent.com/kjam/data-cleaning-101/master/data/sales_data.csv"
)

## 4. Daten untersuchen

In [4]:
sales.head()

,Unnamed: 0,timestamp,city,store_id,sale_number,sale_amount,associate
0,0,2018-09-10 05:00:45,Williamburgh,6,1530,1167.0,Gary Lee
1,1,2018-09-12 10:01:27,Ibarraberg,1,2744,258.0,Daniel Davis
2,2,2018-09-13 12:01:48,Sarachester,2,1908,266.0,Michael Roth
3,3,2018-09-14 20:02:19,Caldwellbury,14,771,-108.0,Michaela Stewart
4,4,2018-09-16 01:03:21,Erikaland,11,1571,-372.0,Mark Taylor


In [5]:
sales.dtypes

Unnamed: 0       int64
timestamp       object
city            object
store_id         int64
sale_number      int64
sale_amount    float64
associate       object
dtype: object

## 5. Schema definieren

In der Spalte `sale_amount` sollen alle Werte zwischen 2,5 and 1450,99 sein.

In [6]:
schema = Schema(
    {
        Required("sale_amount"): All(float, Range(min=2.50, max=1450.99)),
    },
    extra=ALLOW_EXTRA,
)

In [7]:
error_count = 0
for s_id, sale in sales.T.to_dict().items():
    try:
        schema(sale)
    except MultipleInvalid as e:
        logger.warning(
            f"issue with sale: {s_id} ({sale['timestamp']}) - {e}",
        )
        error_count += 1

issue with sale: 3 (2018-09-14 20:02:19) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 4 (2018-09-16 01:03:21) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 5 (2018-09-18 03:04:11) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 6 (2018-09-20 12:04:49) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 7 (2018-09-21 15:05:42) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 10 (2018-09-27 04:07:32) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 13 (2018-10-02 17:08:42) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 15 (2018-10-07 06:09:00) - value must be at least 2.5 for dictionary value @ data['sale_amount']
issue with sale: 19 (2018-10-14 08:10:23) - value must be at least 2.5 for dictionary value @

Um die Elemente einer Spalte als Schlüssel und die Elemente einer anderen Spalte als Werte verwenden zu können, machen wir einfach die gewünschte Spalte zum Index des DataFrame und transponieren sie mit der Funktion `.T()`; siehe auch [pandas.DataFrame.transpose](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.transpose.html).

In [8]:
error_count

69

Aktuell wissen wir jedoch noch nicht, ob

* wir ein falsch definiertes Schema haben
* möglicherweise negative Werte zurückgegeben oder falsch markiert werden
* höhere Werte kombinierte Einkäufe oder Sonderverkäufe sind

## 6. Hinzufügen einer benutzerdefinierten Validierung

In [9]:
def valid_date(fmt="%Y-%m-%d %H:%M:%S"):
    return lambda v: datetime.datetime.strptime(v, fmt).replace(
        tzinfo=datetime.timezone.utc
    )

In [10]:
schema = Schema(
    {
        Required("timestamp"): All(valid_date()),
    },
    extra=ALLOW_EXTRA,
)

In [11]:
error_count = 0
for s_id, sale in sales.T.to_dict().items():
    try:
        schema(sale)
    except MultipleInvalid as e:
        logger.warning(
            f"issue with sale: {s_id} ({sale['timestamp']}) - {e}",
        )
        error_count += 1

In [12]:
error_count

0

## 7. Gültige Datumsstrukturen sind noch keine gültigen Daten

In [13]:
def valid_date(fmt="%Y-%m-%d %H:%M:%S"):
    def validation_func(v):
        try:
            assert datetime.datetime.strptime(v, fmt).replace(
                tzinfo=datetime.timezone.utc
            ) <= datetime.datetime.now(tz=datetime.timezone.utc)
        except AssertionError:
            msg = f"The date is in the future! {v}"
            raise Invalid(msg) from AssertionError

    return validation_func

In [14]:
schema = Schema(
    {
        Required("timestamp"): All(valid_date()),
    },
    extra=ALLOW_EXTRA,
)

In [15]:
error_count = 0
for s_id, sale in sales.T.to_dict().items():
    try:
        schema(sale)
    except MultipleInvalid as e:
        logger.warning(
            f"issue with sale: {s_id} ({sale['timestamp']}) - {e}",
        )
        error_count += 1

In [16]:
error_count

0